In [1]:
import tensorflow as tf

print("TensorFlow:", tf.__version__)
print("Keras:", tf.keras.__version__)

TensorFlow: 2.21.0
Keras: 3.15.1


In [2]:
history = image_model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=15
)

NameError: name 'image_model' is not defined

In [3]:
import os
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2

print("TensorFlow:", tf.__version__)

TensorFlow: 2.21.0


In [4]:
DATASET_DIR = "../data/images"

print(os.path.abspath(DATASET_DIR))
print(os.listdir(DATASET_DIR))

C:\Users\senay\Documents\stray-animal-ai-project\data\images
['Critical', 'Healthy', 'Injured', 'Sick']


In [5]:
class_names = sorted([
    folder for folder in os.listdir(DATASET_DIR)
    if os.path.isdir(os.path.join(DATASET_DIR, folder))
])

print("Classes:", class_names)

Classes: ['Critical', 'Healthy', 'Injured', 'Sick']


In [6]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 4
SEED = 42

train_dataset = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int"
)

validation_dataset = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int"
)

class_names = train_dataset.class_names

print("Classes:", class_names)

Found 16 files belonging to 4 classes.
Using 13 files for training.
Found 16 files belonging to 4 classes.
Using 3 files for validation.
Classes: ['Critical', 'Healthy', 'Injured', 'Sick']


In [7]:
AUTOTUNE = tf.data.AUTOTUNE

train_dataset = train_dataset.shuffle(100)

train_dataset = train_dataset.prefetch(
    buffer_size=AUTOTUNE
)

validation_dataset = validation_dataset.prefetch(
    buffer_size=AUTOTUNE
)

In [8]:
class_names = train_dataset.class_names

AttributeError: '_PrefetchDataset' object has no attribute 'class_names'

In [9]:
import os

DATASET_DIR = "../data/images"

class_names = sorted([
    folder
    for folder in os.listdir(DATASET_DIR)
    if os.path.isdir(os.path.join(DATASET_DIR, folder))
])

print("Classes:", class_names)

Classes: ['Critical', 'Healthy', 'Injured', 'Sick']


In [10]:
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1)
])

base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False

inputs = tf.keras.Input(
    shape=(224, 224, 3)
)

x = data_augmentation(inputs)

x = tf.keras.applications.mobilenet_v2.preprocess_input(x)

x = base_model(x, training=False)

x = layers.GlobalAveragePooling2D()(x)

x = layers.Dropout(0.3)(x)

outputs = layers.Dense(
    len(class_names),
    activation="softmax"
)(x)

image_model = tf.keras.Model(
    inputs=inputs,
    outputs=outputs
)

image_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

image_model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential (Sequential)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide (TrueDivide)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract (Subtract)             │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 4)              │         5,124 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,263,108 (8.63 MB)

 Trainable params: 5,124 (20.02 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [11]:
history = image_model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=15
)

Epoch 1/15


C:\Users\senay\anaconda3\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


4/4 ━━━━━━━━━━━━━━━━━━━━ 12s 1s/step - accuracy: 0.3077 - loss: 1.9183 - val_accuracy: 0.0000e+00 - val_loss: 1.6695
Epoch 2/15
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 196ms/step - accuracy: 0.6154 - loss: 1.1959 - val_accuracy: 0.0000e+00 - val_loss: 1.4454
Epoch 3/15
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 234ms/step - accuracy: 0.6923 - loss: 1.1090 - val_accuracy: 0.3333 - val_loss: 1.3770
Epoch 4/15
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 204ms/step - accuracy: 0.6154 - loss: 0.7884 - val_accuracy: 0.3333 - val_loss: 1.3826
Epoch 5/15
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 224ms/step - accuracy: 0.6923 - loss: 0.7407 - val_accuracy: 0.3333 - val_loss: 1.3599
Epoch 6/15
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 189ms/step - accuracy: 0.6923 - loss: 0.7224 - val_accuracy: 0.3333 - val_loss: 1.3820
Epoch 7/15
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 222ms/step - accuracy: 0.6923 - loss: 0.5997 - val_accuracy: 0.3333 - val_loss: 1.3993
Epoch 8/15
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 218ms/step - accuracy: 1.0000 - loss: 0.3504 - val_accuracy: 0.3333 - val_loss: 1.38

In [12]:
import os

MODEL_DIR = "../models"
os.makedirs(MODEL_DIR, exist_ok=True)

MODEL_PATH = os.path.join(
    MODEL_DIR,
    "animal_image_model.keras"
)

image_model.save(MODEL_PATH)

print("✅ Module 2 model saved successfully!")
print("📁", os.path.abspath(MODEL_PATH))

✅ Module 2 model saved successfully!
📁 C:\Users\senay\Documents\stray-animal-ai-project\models\animal_image_model.keras


In [13]:
import json

class_names = ['Critical', 'Healthy', 'Injured', 'Sick']

CLASS_PATH = os.path.join(
    MODEL_DIR,
    "animal_image_classes.json"
)

with open(CLASS_PATH, "w") as f:
    json.dump(class_names, f)

print("✅ Classes saved!")
print(class_names)

✅ Classes saved!
['Critical', 'Healthy', 'Injured', 'Sick']


In [14]:
import sqlite3

connection = sqlite3.connect(
    "database/stray_animals.db"
)

connection.execute("""
CREATE TABLE IF NOT EXISTS image_predictions (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    image_filename TEXT,
    predicted_health_status TEXT,
    confidence REAL,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
)
""")

connection.commit()
connection.close()

print("image_predictions table created successfully.")

OperationalError: unable to open database file

In [15]:
import sqlite3
import os

# Get the project folder
BASE_DIR = os.path.dirname(
    os.path.dirname(
        os.path.abspath(".")
    )
)

DATABASE_PATH = os.path.join(
    BASE_DIR,
    "database",
    "stray_animals.db"
)

print("Database path:")
print(DATABASE_PATH)

print("\nDatabase exists:")
print(os.path.exists(DATABASE_PATH))

Database path:
C:\Users\senay\Documents\database\stray_animals.db

Database exists:
False


In [16]:
import sqlite3
import os

DATABASE_PATH = r"C:\Users\senay\Documents\stray-animal-ai-project\database\stray_animals.db"

print("Database exists:", os.path.exists(DATABASE_PATH))

connection = sqlite3.connect(DATABASE_PATH)

connection.execute("""
CREATE TABLE IF NOT EXISTS image_predictions (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    image_filename TEXT,
    predicted_health_status TEXT,
    confidence REAL,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
)
""")

connection.commit()
connection.close()

print("✅ image_predictions table created successfully.")

Database exists: True
✅ image_predictions table created successfully.


In [17]:
import sqlite3
import os

DATABASE_PATH = r"C:\Users\senay\Documents\stray-animal-ai-project\database\stray_animals.db"

print("Database exists:", os.path.exists(DATABASE_PATH))

connection = sqlite3.connect(DATABASE_PATH)

connection.execute("""
CREATE TABLE IF NOT EXISTS image_predictions (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    image_filename TEXT,
    predicted_health_status TEXT,
    confidence REAL,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
)
""")

connection.commit()
connection.close()

print("✅ image_predictions table created successfully.")

Database exists: True
✅ image_predictions table created successfully.
